In [17]:
### import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
import pickle
import json
import matplotlib.pyplot as plt
from glob import glob
import time
import copy
from tqdm import tqdm
import torch.nn.functional as F
import time
from transformers import BertLMHeadModel, BartTokenizer, BartForConditionalGeneration, BartConfig, BartForSequenceClassification, BertTokenizer, BertConfig, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification, PegasusForConditionalGeneration, PegasusTokenizer, T5Tokenizer, T5ForConditionalGeneration, BertGenerationDecoder
from data_singleword import otago_dataset
from model_decoding import BrainTranslator, BrainTranslatorNaive, T5Translator
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from rouge import Rouge
from config import get_config
import evaluate
from evaluate import load
import os
import json 

metric = evaluate.load("sacrebleu")
cer_metric = load("cer")
wer_metric = load("wer")

In [13]:
args = {
    "checkpoint_path": "/projects/open_vocab/saved_models/last/randinit_task1_finetune_T5Translator_2steptraining_b32_15_25_0.005_0.005_unique_sent_clean.pt",
    "config_path": "train_config_t5.json",
    "test_input": "clean",
    "train_input": "clean",
    "cuda": "cuda:0",
}

In [18]:
batch_size = 1
test_input = args['test_input']
print("test_input is:", test_input)
train_input = args['train_input']
print("train_input is:", train_input)
''' load training config'''
training_config = json.load(open(args['config_path']))


subject_choice = training_config['subjects']
print(f'[INFO]subjects: {subject_choice}')
eeg_type_choice = training_config['eeg_type']
print(f'[INFO]eeg type: {eeg_type_choice}')
bands_choice = training_config['eeg_bands']
print(f'[INFO]using bands: {bands_choice}')

dataset_setting = 'unique_sent'

task_name = training_config['task_name']
model_name = training_config['model_name']


if test_input == 'EEG' and train_input=='EEG':
    print("EEG and EEG")
    output_all_results_path = f'./results/{task_name}-{model_name}-all_decoding_results.txt'
    score_results = f'./score_results/{task_name}-{model_name}.txt'
else:
    output_all_results_path = f'./results/{task_name}-{model_name}-{train_input}_{test_input}-all_decoding_results.txt'
    score_results = f'./score_results/{task_name}-{model_name}-{train_input}_{test_input}.txt'


''' set random seeds '''
seed_val = 20 #500
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

''' set up device '''
# use cuda
if torch.cuda.is_available():  
    dev = 0
else:  
    dev = "cpu"
# CUDA_VISIBLE_DEVICES=0,1,2,3  
device = torch.device(dev)
print(f'[INFO]using device {dev}')

''' set up dataloader '''
whole_dataset_dicts = []
if 'task1' in task_name:
    dataset_path_task1 = 'eeg/processed_data/pickle/single_word_classification.pickle'
    with open(dataset_path_task1, 'rb') as handle:
        whole_dataset_dicts.append(pickle.load(handle))
if 'task2' in task_name:
    dataset_path_task2 = 'eeg/processed_data/pickle/single_word_classification_read.pickle'
    with open(dataset_path_task2, 'rb') as handle:
        whole_dataset_dicts.append(pickle.load(handle))
print()

if model_name in ['BrainTranslator','BrainTranslatorNaive']:
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

elif model_name == 'PegasusTranslator':
    tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-xsum')

elif model_name == 'T5Translator':
    tokenizer = T5Tokenizer.from_pretrained("t5-large")
    # tokenizer.set_prefix_tokens(language='english')

# test dataset
test_set = otago_dataset(whole_dataset_dicts, 'test', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=test_input)

dataset_sizes = {"test_set":len(test_set)}
print('[INFO]test_set size: ', len(test_set))

# dataloaders
test_dataloader = DataLoader(test_set, batch_size = batch_size, shuffle=False, num_workers=4)

dataloaders = {'test':test_dataloader}

''' set up model '''
checkpoint_path = args['checkpoint_path']

if model_name == 'BrainTranslator':
    pretrained_bart = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
    model = BrainTranslator(pretrained_bart, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'BrainTranslatorNaive':
    pretrained_bart = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
    model = BrainTranslatorNaive(pretrained_bart, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'BertGeneration':
    pretrained = BertGenerationDecoder.from_pretrained('google-bert/bert-large-uncased', is_decoder = True)
    model = BrainTranslator(pretrained, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)
    
elif model_name == 'PegasusTranslator':
    pretrained = PegasusForConditionalGeneration.from_pretrained('google/pegasus-xsum')
    model = BrainTranslator(pretrained, in_feature = 105*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)

elif model_name == 'T5Translator':
    pretrained = T5ForConditionalGeneration.from_pretrained("t5-large")
    model = T5Translator(pretrained, in_feature = 64*len(bands_choice), decoder_embedding_size = 1024, additional_encoder_nhead=8, additional_encoder_dim_feedforward = 2048)


state_dict = torch.load(checkpoint_path)
new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(new_state_dict)

'''
if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(torch.load(checkpoint_path))
else:
    model.load_state_dict(torch.load(checkpoint_path))
'''

# model.load_state_dict(torch.load(checkpoint_path))
model.to(device)

criterion = nn.CrossEntropyLoss()


test_input is: clean
train_input is: clean
[INFO]subjects: ALL
[INFO]eeg type: GD
[INFO]using bands: ['_t1', '_t2', '_a1', '_a2', '_b1', '_b2', '_g1', '_g2']
[INFO]using device 0


FileNotFoundError: [Errno 2] No such file or directory: 'eeg/processed_data/pickle/single_word_classification.pickle'

In [19]:
def eval_model(dataloaders, device, tokenizer, criterion, model, output_all_results_path = './results/temp.txt' , score_results='./score_results/task.txt'):
    # modified from: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
    start_time = time.time()
    model.eval()   # Set model to evaluate mode
    
    target_tokens_list = []
    target_string_list = []
    pred_tokens_list = []
    pred_string_list = []
    pred_tokens_list_previous = []
    pred_string_list_previous = []
    pred_top5_tokens_list = []
    pred_top10_tokens_list = []  # Added for top-10
    pred_top20_tokens_list = []  # Added for top-20

    with open(output_all_results_path,'w') as f:
        for input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels in tqdm(dataloaders['test']):

            print("input_embeddings", input_embeddings.shape, input_embeddings)

            
            # load in batch
            input_embeddings_batch = input_embeddings.to(device).float() # B, 56, 840
            input_masks_batch = input_masks.to(device) # B, 56
            target_ids_batch = target_ids.to(device) # B, 56
            input_mask_invert_batch = input_mask_invert.to(device) # B, 56
            
            target_ids_batch = target_ids_batch[:, :1] # first token only
            
            target_tokens = tokenizer.convert_ids_to_tokens(target_ids_batch[0].tolist(), skip_special_tokens = True)
            target_string = tokenizer.decode(target_ids_batch[0], skip_special_tokens = True)
            print('target ids tensor:',target_ids_batch[0])
            print('target ids:',target_ids_batch[0].tolist())
            print('target tokens:',target_tokens)
            # print('target string:',target_strininvert.to(device)) # B, 56
            print('target string:', target_string)
            
            f.write(f'target string: {target_string}\n')

            # add to list for later calculate bleu metric
            target_tokens_list.append([target_tokens])
            target_string_list.append(target_string)

            # print("target_tokens_list", target_tokens_list)
            # print("target_string_list", target_string_list)
            
            """replace padding ids in target_ids with -100"""
            target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100 

            # target_ids_batch_label = target_ids_batch.clone().detach()
            # target_ids_batch_label[target_ids_batch_label == tokenizer.pad_token_id] = -100

            # Original code 
            seq2seqLMoutput = model(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch) # (batch, time, n_class)
            logits_previous = seq2seqLMoutput.logits
            probs_previous = logits_previous[0].softmax(dim = 1)
            
            # Get top-5 predictions
            values_previous_5, predictions_previous_5 = probs_previous.topk(5)
            top_5_tokens = predictions_previous_5.flatten()
            print("top 5 prob tokens:", top_5_tokens.tolist())
            print("top 5 prob strings:", tokenizer.decode(top_5_tokens))
            f.write(f'top 5 prob strings: {tokenizer.decode(top_5_tokens)}\n')
            pred_top5_tokens_list.append(top_5_tokens.tolist())
            
            # Get top-10 predictions (added)
            values_previous_10, predictions_previous_10 = probs_previous.topk(10)
            top_10_tokens = predictions_previous_10.flatten()
            print("top 10 prob tokens:", top_10_tokens.tolist())
            print("top 10 prob strings:", tokenizer.decode(top_10_tokens))
            f.write(f'top 10 prob strings: {tokenizer.decode(top_10_tokens)}\n')
            pred_top10_tokens_list.append(top_10_tokens.tolist())
            
            # Get top-20 predictions (added)
            values_previous_20, predictions_previous_20 = probs_previous.topk(20)
            top_20_tokens = predictions_previous_20.flatten()
            print("top 20 prob tokens:", top_20_tokens.tolist())
            print("top 20 prob strings:", tokenizer.decode(top_20_tokens))
            f.write(f'top 20 prob strings: {tokenizer.decode(top_20_tokens)}\n')
            pred_top20_tokens_list.append(top_20_tokens.tolist())
            
            # Original top-1 prediction
            values_previous, predictions_previous = probs_previous.topk(1)
            predictions_previous = torch.squeeze(predictions_previous)
            predicted_string_previous = remove_text_after_token(tokenizer.decode(predictions_previous).split('</s></s>')[0].replace('<s>',''))
            f.write(f'predicted string with tf: {predicted_string_previous}\n')
            print("predicted token with tf:", predictions_previous)
            print('predicted string with tf:',predicted_string_previous)
            predictions_previous = predictions_previous.tolist()
            if isinstance(predictions_previous, int):
                truncated_prediction_previous = [predictions_previous]
            else:
                truncated_prediction_previous = []
                for t in predictions_previous:
                    if t != tokenizer.eos_token_id:
                        truncated_prediction_previous.append(t)
                    else:
                        break
            pred_tokens_previous = tokenizer.convert_ids_to_tokens(truncated_prediction_previous, skip_special_tokens = False)
            pred_tokens_list_previous.append(pred_tokens_previous)
            pred_string_list_previous.append(predicted_string_previous)

            # print("pred_tokens_list_previous", pred_tokens_list_previous)
            # print("pred_string_list_previous", pred_string_list_previous)

            # Modify code
            predictions=model.generate(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch,
                                       # max_length=1,
                                       # num_beams=5,
                                       do_sample=False,
                                       # repetition_penalty= 5.0,
                                       # no_repeat_ngram_size = 2
                                       # num_beams=5,encoder_no_repeat_ngram_size =1,
                                       # do_sample=True, top_k=15,temperature=0.5,num_return_sequences=5,
                                       # early_stopping=True
                                       )

            print("predictions", predictions)
            
            pred_tokens = predictions[:, -1]
            predicted_string = tokenizer.decode(pred_tokens)
            
            # predicted_string=tokenizer.batch_decode(predictions, skip_special_tokens=False)[0]
            # predicted_string=predicted_string.squeeze()
            
            predictions=tokenizer.encode(predicted_string)
            # print('predicted string:',predicted_string)
            f.write(f'predicted string: {predicted_string}\n')
            f.write(f'################################################\n\n\n')
            
            print('predicted_string:',predicted_string)
            print('predicted tokens:',pred_tokens)
            pred_tokens_list.append(pred_tokens)
            pred_string_list.append(predicted_string)
            # pred_tokens_list.extend(pred_tokens)
            # pred_string_list.extend(predicted_string)
            print('################################################')
            # print()
    # print(f"pred_string_list : {pred_string_list}")
    
    """ calculate corpus bleu score """
    weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]
    corpus_bleu_scores = []
    corpus_bleu_scores_previous = []
    for weight in weights_list:
        # print('weight:',weight)
        corpus_bleu_score = corpus_bleu(target_tokens_list, pred_tokens_list, weights = weight)
        corpus_bleu_score_previous = corpus_bleu(target_tokens_list, pred_tokens_list_previous, weights = weight)
        corpus_bleu_scores.append(corpus_bleu_score)
        corpus_bleu_scores_previous.append(corpus_bleu_score_previous)
        print(f'corpus BLEU-{len(list(weight))} score:', corpus_bleu_score)
        print(f'corpus BLEU-{len(list(weight))} score with tf:', corpus_bleu_score_previous)


    """ calculate sacre bleu score """
    
    reference_list = [[item] for item in target_string_list]

    #print(f'ref: {reference_list}')
    #print(f'pred: {prediction_list}')
    sacre_blue = metric.compute(predictions=pred_string_list, references=reference_list)
    sacre_blue_previous = metric.compute(predictions=pred_string_list_previous, references=reference_list)
    print("sacreblue score: ", sacre_blue, '\n')
    print("sacreblue score with tf: ", sacre_blue_previous)


    print()
    """ calculate rouge score """
    rouge = Rouge()

    try:
        rouge_scores = rouge.get_scores(pred_string_list, target_string_list, avg = True, ignore_empty=True)
    except ValueError as e:
        rouge_scores = 'Hypothesis is empty'

    try:
        rouge_scores_previous = rouge.get_scores(pred_string_list_previous, target_string_list, avg = True, ignore_empty=True)
    except ValueError as e:
        rouge_scores_previous = 'Hypothesis is empty'
    print()


    print()
    """ calculate WER score """
    #wer = WordErrorRate()
    wer_scores = wer_metric.compute(predictions=pred_string_list, references=target_string_list)
    wer_scores_previous = wer_metric.compute(predictions=pred_string_list_previous, references=target_string_list)
    print("WER score:", wer_scores)
    print("WER score with tf:", wer_scores_previous)
    

    """ calculate CER score """
    cer_scores = cer_metric.compute(predictions=pred_string_list, references=target_string_list)
    cer_scores_previous = cer_metric.compute(predictions=pred_string_list_previous, references=target_string_list)
    print("CER score:", cer_scores)
    print("CER score with tf:", cer_scores_previous)

    """ calculate accuracy """
    total_tokens = 0
    correct_tokens = 0
    
    for target_tokens, pred_tokens in zip(target_tokens_list, pred_tokens_list):
        # flatten if tokens are wrapped in extra list: [['▁help']] -> ['▁help']
        target_tokens_flat = target_tokens[0] if isinstance(target_tokens[0], list) else target_tokens
        pred_tokens_flat = pred_tokens[0] if isinstance(pred_tokens[0], list) else pred_tokens
        
        # compare token by token
        for t, p in zip(target_tokens_flat, pred_tokens_flat):
            total_tokens += 1
            if t == p:
                correct_tokens += 1
    
    token_accuracy = correct_tokens / total_tokens
    print(f"First token accuracy: {correct_tokens}/{total_tokens} {token_accuracy}")

    total_tokens = 0
    correct_tokens = 0
    for target_tokens, pred_tokens in zip(target_tokens_list, pred_tokens_list_previous):
        # flatten if tokens are wrapped in extra list: [['▁help']] -> ['▁help']
        target_tokens_flat = target_tokens[0] if isinstance(target_tokens[0], list) else target_tokens
        pred_tokens_flat = pred_tokens[0] if isinstance(pred_tokens[0], list) else pred_tokens
        
        # compare token by token
        for t, p in zip(target_tokens_flat, pred_tokens_flat):
            total_tokens += 1
            if t == p:
                correct_tokens += 1
    
    token_accuracy_tf = correct_tokens / total_tokens
    print(f"First token accuracy with tf: {correct_tokens}/{total_tokens} {token_accuracy_tf}")

    
    # Compute top-5 accuracy
    total_tokens = 0
    correct_tokens = 0
    
    for target_tokens, pred_tokens in zip(target_tokens_list, pred_top5_tokens_list):
        # flatten target string
        target_token = target_tokens[0][0]  # ['▁help'] -> '▁help'
        # convert target token to token ID
        target_id = tokenizer(target_token).input_ids[0]
        
        total_tokens += 1
        if target_id in pred_tokens:  # top-5 check
            correct_tokens += 1
    
    top5_accuracy = correct_tokens / total_tokens
    print(f"Top-5 accuracy: {correct_tokens}/{total_tokens} = {top5_accuracy:.4f}")
    
    # Compute top-10 accuracy (added)
    total_tokens = 0
    correct_tokens = 0
    
    for target_tokens, pred_tokens in zip(target_tokens_list, pred_top10_tokens_list):
        # flatten target string
        target_token = target_tokens[0][0]  # ['▁help'] -> '▁help'
        # convert target token to token ID
        target_id = tokenizer(target_token).input_ids[0]
        
        total_tokens += 1
        if target_id in pred_tokens:  # top-10 check
            correct_tokens += 1
    
    top10_accuracy = correct_tokens / total_tokens
    print(f"Top-10 accuracy: {correct_tokens}/{total_tokens} = {top10_accuracy:.4f}")
    
    # Compute top-20 accuracy (added)
    total_tokens = 0
    correct_tokens = 0
    
    for target_tokens, pred_tokens in zip(target_tokens_list, pred_top20_tokens_list):
        # flatten target string
        target_token = target_tokens[0][0]  # ['▁help'] -> '▁help'
        # convert target token to token ID
        target_id = tokenizer(target_token).input_ids[0]
        
        total_tokens += 1
        if target_id in pred_tokens:  # top-20 check
            correct_tokens += 1
    
    top20_accuracy = correct_tokens / total_tokens
    print(f"Top-20 accuracy: {correct_tokens}/{total_tokens} = {top20_accuracy:.4f}")
    

    end_time = time.time()
    print(f"Evaluation took {(end_time-start_time)/60} minutes to execute.")

     # score_results (only fix teacher-forcing)
    file_content = [
    f'corpus_bleu_score = {corpus_bleu_scores}',
    f'sacre_blue_score = {sacre_blue}',
    f'rouge_scores = {rouge_scores}',
    f'wer_scores = {wer_scores}',
    f'cer_scores = {cer_scores}',
    f'corpus_bleu_score_with_tf = {corpus_bleu_scores_previous}',
    f'sacre_blue_score_with_tf = {sacre_blue_previous}',
    f'rouge_scores_with_tf = {rouge_scores_previous}',
    f'wer_scores_with_tf = {wer_scores_previous}',
    f'cer_scores_with_tf = {cer_scores_previous}',
    f'token_accuracy = {token_accuracy}',
    f'token_accuracy_tf = {token_accuracy_tf}',
    f'top5_accuracy = {top5_accuracy}',
    f'top10_accuracy = {top10_accuracy}',  # Added
    f'top20_accuracy = {top20_accuracy}',  # Added
    ]

    os.makedirs(os.path.dirname(score_results), exist_ok=True)
    
    with open(score_results, "a") as file_results:
        for line in file_content:
            if isinstance(line, list):
                for item in line:
                    file_results.write(str(item) + "\n")
            else:
                file_results.write(str(line) + "\n")

In [16]:
eval_model(dataloaders, device, tokenizer, criterion, model, output_all_results_path = output_all_results_path, score_results=score_results)


  0%|          | 0/440 [00:00<?, ?it/s]

input_embeddings torch.Size([1, 2, 512]) tensor([[[ 2.8618e-08,  1.4401e-08, -1.7544e-09,  ...,  2.3042e-08,
           1.5490e-08,  1.0994e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]])
target ids tensor: tensor([199], device='cuda:0')
target ids: [199]
target tokens: ['▁help']
target string: help
input_embeddings_batch shape torch.Size([1, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([1, 2, 512])
top 5 prob tokens: [1282, 4273, 150, 207, 21820]
top 5 prob strings: bad yes no good hello
top 10 prob tokens: [1282, 4273, 150, 207, 21820, 281, 199, 1190, 2049, 1095]
top 10 prob strings: bad yes no good hello go help stop thanks happy
top 20 prob tokens: [1282, 4273, 150, 207, 21820, 281, 199, 1190, 2049, 1095, 2763, 248, 129, 7102, 23281, 17945, 1245, 278, 653, 8]
top 20 prob strings: bad yes no good hello go help stop thanks happy thank great get hi goodbye yeah nice don try the
predicted token with tf: tensor(1282, 

/home/moupe847/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
  0%|          | 1/440 [00:01<09:08,  1.25s/it]

predictions tensor([[   0, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282,
         1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282]],
       device='cuda:0')
predicted_string: bad
predicted tokens: tensor([1282], device='cuda:0')
################################################
input_embeddings torch.Size([1, 2, 512]) tensor([[[ 6.5848e-08,  5.0569e-08,  2.6068e-08,  ..., -2.4103e-08,
          -1.0563e-08, -1.5507e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]])
target ids tensor: tensor([1282], device='cuda:0')
target ids: [1282]
target tokens: ['▁bad']
target string: bad
input_embeddings_batch shape torch.Size([1, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([1, 2, 512])
top 5 prob tokens: [1282, 4273, 150, 207, 21820]
top 5 prob strings: bad yes no good hello
top 10 prob tokens: [1282, 4273, 150, 207, 21820, 281, 199, 1190, 2049, 1095]
top 10 prob strings: bad yes no good hello go help stop

  0%|          | 2/440 [00:01<05:59,  1.22it/s]

...

predictions tensor([[   0, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282,
         1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282]],
       device='cuda:0')
predicted_string: bad
predicted tokens: tensor([1282], device='cuda:0')
################################################
input_embeddings torch.Size([1, 2, 512]) tensor([[[ 5.6117e-08,  1.9868e-08, -3.6518e-08,  ..., -1.0749e-08,
          -2.7590e-09, -7.6733e-09],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]])
target ids tensor: tensor([21820], device='cuda:0')
target ids: [21820]
target tokens: ['▁hello']
target string: hello
input_embeddings_batch shape torch.Size([1, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([1, 2, 512])
top 5 prob tokens: [1282, 4273, 150, 207, 21820]
top 5 prob strings: bad yes no good hello
top 10 prob tokens: [1282, 4273, 150, 207, 21820, 281, 199, 1190, 2049, 1095]
top 10 prob strings: bad yes no good hello go hel

 10%|█         | 44/440 [00:22<03:13,  2.05it/s]

predictions tensor([[   0, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282,
         1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282, 1282]],
       device='cuda:0')
predicted_string: bad
predicted tokens: tensor([1282], device='cuda:0')
################################################
input_embeddings torch.Size([1, 2, 512]) tensor([[[-9.0980e-08, -1.0585e-07, -1.1202e-07,  ..., -3.4187e-09,
          -6.2674e-09, -1.0673e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]])
target ids tensor: tensor([2049], device='cuda:0')
target ids: [2049]
target tokens: ['▁thanks']
target string: thanks
input_embeddings_batch shape torch.Size([1, 2, 512])
[DEBUG] EEG input to encoder: torch.Size([1, 2, 512])
top 5 prob tokens: [1282, 4273, 150, 207, 21820]
top 5 prob strings: bad yes no good hello
top 10 prob tokens: [1282, 4273, 150, 207, 21820, 281, 199, 1190, 2049, 1095]
top 10 prob strings: bad yes no good hello go hel

 10%|█         | 44/440 [00:22<03:23,  1.95it/s]


KeyboardInterrupt: 